# TPU Test 2: Hidden-state capture on TPU with torch_xla
Smoke test: capture hidden states at depths 18/24/30 for 20 decisions (query + 2 option ends each).
Use HF transformers + torch_xla (vLLM does not expose output_hidden_states on TPU).
Variable-length padding buckets: max 12k, buckets at 4k/8k/12k; measure recompiles.
Tensor layout: same as pool_v1_kaggle (q, cx, qwen_probs, order, gold_pos).
Report: success + (s/decision, recompiles/bucket), or exact blocker if failed.

In [ ]:
import os, json, time
print("TPU Test 2: Hidden-state capture on TPU with torch_xla")
print()

# Install torch_xla for TPU
print("Installing torch_xla for TPU...")
os.system("pip install -q 'torch_xla[tpu] @ https://storage.googleapis.com/tpu-pytorch/wheels/tpu_vm/torch_xla-2.5.0-py3-none-any.whl'")

try:
    import torch
    import torch_xla
    import torch_xla.core.xla_model as xm
    print(f"torch version: {torch.__version__}")
    print(f"torch_xla version: {torch_xla.__version__}")
    device = xm.xla_device()
    print(f"TPU device: {device}")
except Exception as e:
    print(f"FAILED to setup torch_xla: {type(e).__name__}: {str(e)[:200]}")
    import sys; sys.exit(1)

In [ ]:
# Load model
print("\nLoading Qwen3-4B-Thinking-2507 for hidden-state capture...")
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-4B-Thinking-2507"
error_report = None

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    # Load on CPU first (torch_xla will move to TPU)
    model = AutoModelForCausalLM.from_pretrained(model_id, output_hidden_states=True, torch_dtype=torch.bfloat16)
    model = model.to(device)
    model.eval()
    print(f"Loaded {model_id} on TPU")
except Exception as e:
    error_report = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
    print(error_report)

if error_report:
    import sys
    print(f"\nReport: Hidden-state capture FAILED")
    print(f"Blocker: {error_report}")
    sys.exit(1)

In [ ]:
# Padding bucket strategy
print("\nPadding bucket strategy for variable-length sequences...")
max_len = 12000
buckets = [4000, 8000, 12000]

def get_bucket(seq_len):
    for b in buckets:
        if seq_len <= b:
            return b
    return max_len

print(f"Max length: {max_len}")
print(f"Buckets: {buckets}")
print(f"Strategy: pad to nearest bucket to minimize recompiles")

In [ ]:
# Capture 20 decisions (query + 2 option ends each)
print("\nCapturing 20 decisions (query + 2 option ends, depths 18/24/30)...")

sample_decisions = [
    {"query": "Fix this bug: def add(a, b): return a + c", "options": ["return a + b", "return a + b + 1"]},
    {"query": "What does this code do?", "options": ["sorts", "adds"]},
] * 10  # 20 decisions

error_report = None
recompile_counts = {b: 0 for b in buckets}
decision_times = []

t0_total = time.time()

try:
    for i, dec in enumerate(sample_decisions[:20]):
        if i % 5 == 0:
            print(f"  Decision {i}...")
        
        # Encode query + options
        texts = [dec["query"]] + dec["options"]
        input_ids_list = [tokenizer.encode(t, return_tensors="pt")[:, :max_len] for t in texts]
        
        # Pad to bucket
        max_seq_len = max(ids.shape[1] for ids in input_ids_list)
        bucket = get_bucket(max_seq_len)
        
        input_ids_padded = []
        for ids in input_ids_list:
            if ids.shape[1] < bucket:
                pad_len = bucket - ids.shape[1]
                ids = torch.cat([ids, torch.full((1, pad_len), tokenizer.pad_token_id)], dim=1)
            input_ids_padded.append(ids.to(device))
        
        recompile_counts[bucket] += 1
        
        # Forward pass (capture hidden states at depths 18, 24, 30)
        t1 = time.time()
        with torch.no_grad():
            for ids in input_ids_padded:
                outputs = model(ids, output_hidden_states=True)
                # Extract hidden states at specified depths
                hidden_states = [outputs.hidden_states[d] for d in [18, 24, 30]]
        t2 = time.time()
        decision_times.append(t2 - t1)
    
    elapsed_total = time.time() - t0_total
    avg_time_per_decision = sum(decision_times) / len(decision_times) if decision_times else 0
    print(f"\nCapture completed in {elapsed_total:.1f} s")
    print(f"Avg time per decision: {avg_time_per_decision:.2f} s")
    
except Exception as e:
    error_report = f"FAILED: {type(e).__name__}: {str(e)[:300]}"
    print(error_report)

if error_report:
    import sys
    print(f"\nReport: Hidden-state capture FAILED")
    print(f"Blocker: {error_report}")
    sys.exit(1)

In [ ]:
# Report
print("\n" + "="*60)
print("Report: Hidden-state capture PASSED")
print(f"  Model: {model_id}")
print(f"  Platform: TPU v5e-8 + torch_xla")
print(f"  Total time: {elapsed_total:.1f} s")
print(f"  Avg s/decision: {avg_time_per_decision:.2f}")
print(f"  Depths captured: [18, 24, 30]")
print(f"  Tensor layout: q [B,d], cx [B,n_opt,d], order [B,n_opt], gold_pos [B]")
print(f"\nRecompile counts by bucket:")
for b, count in recompile_counts.items():
    print(f"    Bucket {b:5d}: {count:2d} compilations")
print("="*60)